<a href="https://colab.research.google.com/github/mohiuddin2806-lang/machine-learning-/blob/main/week3mohi.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip3 install pyspark

In [2]:
#initialize SparkSession and installed Required Libraries
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegression_spark") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()


spark

In [3]:
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")


Spark UI available at: http://9b93ff643a6d:4040


In [4]:
spark.sparkContext.setLogLevel("INFO")

In [5]:
import psutil
print(f"CPU Usage: {psutil.cpu_percent()}%")
print(f"Memory Usage: {psutil.virtual_memory().percent}%")

CPU Usage: 51.4%
Memory Usage: 11.7%


In [9]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
# Import necessary libraries for SparkSession
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator

# Initialize SparkSession
spark = SparkSession.builder \
                    .appName("LinearRegression_spark") \
                    .master("local[*]") \
                    .config("spark.executor.memory", "4g") \
                    .config("spark.driver.memory", "2g") \
                    .config("spark.executor.cores", "2") \
                    .config("spark.sql.inMemoryColumnarStorage.compressed", "true") \
                    .getOrCreate()

# Display the SparkSession object to confirm initialization
spark

After running the above cell, you should see a `SparkSession` object printed, indicating that Spark is ready for use.

In [9]:
# Load the data from a CSV file
df = spark.read.csv("/content/property.csv", header=True, inferSchema=True)

# get familiar with data
df.show()

# more info
print("Total Records",df.count())
print("Total Partitions ",df.rdd.getNumPartitions())

+--------------+------------+-------------+----------+--------+------------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|             Price|
+--------------+------------+-------------+----------+--------+------------------+
|          1360|           2|            3|      1953|    7860| 303948.1373854071|
|          4272|           3|            3|      1997|    5292| 860386.2685075302|
|          3592|           4|            1|      1983|    9723| 734389.7538956215|
|           966|           6|            1|      1903|    4086| 226448.8070714377|
|          4926|           6|            4|      1944|    1081|1022486.2616704078|
|          3944|           6|            2|      1938|    3542| 845638.1354384426|
|          3671|           2|            1|      1963|    5105| 748779.2192281872|
|          3419|           4|            2|      1925|    5448| 743007.2614135538|
|           630|           2|            2|      2012|    3204| 135656.4528785377|
|   

In [10]:
# show Schema,Prints the structure of the dataset
df.printSchema()


root
 |-- Square_Footage: integer (nullable = true)
 |-- Num_Bedrooms: integer (nullable = true)
 |-- Num_Bathrooms: integer (nullable = true)
 |-- Year_Built: integer (nullable = true)
 |-- Lot_Size: integer (nullable = true)
 |-- Price: double (nullable = true)



In [11]:
# show Schema,Prints the structure of the dataset
df.printSchema()

root
 |-- Square_Footage: integer (nullable = true)
 |-- Num_Bedrooms: integer (nullable = true)
 |-- Num_Bathrooms: integer (nullable = true)
 |-- Year_Built: integer (nullable = true)
 |-- Lot_Size: integer (nullable = true)
 |-- Price: double (nullable = true)



In [12]:
#Statistical Analysis
df.describe().show()

+-------+-----------------+-----------------+------------------+-----------------+-----------------+------------------+
|summary|   Square_Footage|     Num_Bedrooms|     Num_Bathrooms|       Year_Built|         Lot_Size|             Price|
+-------+-----------------+-----------------+------------------+-----------------+-----------------+------------------+
|  count|          1000000|          1000000|           1000000|          1000000|          1000000|           1000000|
|   mean|      2750.657104|         3.501114|          2.500439|       1960.52736|      5502.373911| 581839.6653163614|
| stddev|1298.569362387213|1.708173784151257|1.1178528780094728|35.21780350510348|2598.885882999355|260685.36722644986|
|    min|              500|                1|                 1|             1900|             1000| 51495.71116919513|
|    max|             4999|                6|                 4|             2021|             9999|1123219.4691521737|
+-------+-----------------+-------------

In [14]:
import psutil
print(f"CPU Usage after openig the csv file: {psutil.cpu_percent()}%")
print(f"Memory Usage after csv file: {psutil.virtual_memory().percent}%")

CPU Usage after openig the csv file: 27.6%
Memory Usage after csv file: 17.5%


In [15]:
# convert categorical column into numbers
from pyspark.ml.feature import StringIndexer
indexer = StringIndexer(inputCol = 'sex', outputCol = 'gender')
#df = indexer.fit(df).transform(df)
#df.show(30)

In [17]:
from pyspark.sql.functions import col
from pyspark.sql.types import DoubleType

# 'df_cleaned_price' was not defined. Assuming it should be derived from 'df'.
# Also, 'Price_double' is used in the filter, so let's create it from the 'Price' column.
# The 'Price' column in 'df' is already of type double.
df_cleaned_price = df.withColumn("Price_double", col("Price").cast(DoubleType()))

# Filter rows where the newly casted Price_double is null (due to malformed string data)
null_price_rows = df_cleaned_price.filter(col("Price_double").isNull())

print(f"Total rows with invalid/null Price: {null_price_rows.count()}")
print("Preview of records with invalid Price values:")
null_price_rows.show(10)

Total rows with invalid/null Price: 0
Preview of records with invalid Price values:
+--------------+------------+-------------+----------+--------+-----+------------+
|Square_Footage|Num_Bedrooms|Num_Bathrooms|Year_Built|Lot_Size|Price|Price_double|
+--------------+------------+-------------+----------+--------+-----+------------+
+--------------+------------+-------------+----------+--------+-----+------------+



In [18]:
# 1. Clean the data by dropping rows with any null values in our selected columns
# This ensures the VectorAssembler doesn't fail on nulls.
features = ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size']
target = 'Price_double'

cleaned_df = df_cleaned_price.dropna(subset=features + [target])

# 2. Assemble features into a single vector column
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols=features, outputCol="features")

# Transform the data
ml_data = assembler.transform(cleaned_df).select("features", target)

# Display the prepared data for machine learning
print(f"Original row count: {df_cleaned_price.count()}")
print(f"Cleaned row count: {cleaned_df.count()}")
ml_data.show(5)

Original row count: 1000000
Cleaned row count: 1000000
+--------------------+------------------+
|            features|      Price_double|
+--------------------+------------------+
|[1360.0,2.0,3.0,1...| 303948.1373854071|
|[4272.0,3.0,3.0,1...| 860386.2685075302|
|[3592.0,4.0,1.0,1...| 734389.7538956215|
|[966.0,6.0,1.0,19...| 226448.8070714377|
|[4926.0,6.0,4.0,1...|1022486.2616704078|
+--------------------+------------------+
only showing top 5 rows


In [19]:
from pyspark.ml.regression import LinearRegression
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import VectorAssembler

# Split the data into training and test sets
train_data, test_data = cleaned_df.randomSplit([0.8, 0.2], seed=42)

# Define feature combinations
feature_sets = {
    "Full Set": ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms', 'Year_Built', 'Lot_Size'],
    "Size and Age": ['Square_Footage', 'Year_Built', 'Lot_Size'],
    "Interior Features": ['Square_Footage', 'Num_Bedrooms', 'Num_Bathrooms']
}

evaluator_rmse = RegressionEvaluator(labelCol="Price_double", predictionCol="prediction", metricName="rmse")
evaluator_r2 = RegressionEvaluator(labelCol="Price_double", predictionCol="prediction", metricName="r2")

results = []

for name, cols in feature_sets.items():
    # Assemble features for this specific set
    vassembler = VectorAssembler(inputCols=cols, outputCol="current_features")

    # Transform train/test
    train_transformed = vassembler.transform(train_data).select("current_features", "Price_double")
    test_transformed = vassembler.transform(test_data).select("current_features", "Price_double")

    # Train Linear Regression Model
    lr = LinearRegression(featuresCol="current_features", labelCol="Price_double")
    lr_model = lr.fit(train_transformed)

    # Make Predictions
    predictions = lr_model.transform(test_transformed)

    # Evaluate
    rmse = evaluator_rmse.evaluate(predictions)
    r2 = evaluator_r2.evaluate(predictions)

    results.append({"Model Name": name, "Features": cols, "RMSE": rmse, "R2": r2})

# Display Results
import pandas as pd
results_df = pd.DataFrame(results)
display(results_df)

,Model Name,Features,RMSE,R2
0,Full Set,"[Square_Footage, Num_Bedrooms, Num_Bathrooms, ...",20006.733452,0.994098
1,Size and Age,"[Square_Footage, Year_Built, Lot_Size]",21979.357871,0.992877
2,Interior Features,"[Square_Footage, Num_Bedrooms, Num_Bathrooms]",20328.879759,0.993906
